# 01 - LSTM Pakcoy: klasifikasi tahap tumbuh + prediksi 4 hari ke depan

**Tujuan**
1. Klasifikasi tahap tumbuh (`masa_pertumbuhan`, `mendekati_panen`, `siap_panen`) dari 6 pembacaan sensor terakhir.
2. Prediksi 12 pembacaan berikutnya (= 4 hari, 3 pembacaan per hari) untuk kelembapan, suhu, dan maturity.

**Data**: sheet `Data_Training` (30 siklus, latih/validasi/uji dibagi per siklus lewat kolom `Split`).
Sheet `Sheet` (data asli, 1 siklus) dipakai sebagai uji tambahan.

**Sebelum mulai**: pastikan file Excel ada di `MyDrive/pakcoy-project/data/`. Jalankan sel dari atas ke bawah (Runtime > Run all).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

BASE = '/content/drive/MyDrive/pakcoy-project'
DATA_XLSX = f'{BASE}/data/dataset_pakcoy_monitoring_40hari.xlsx'
MODELS = f'{BASE}/models'
os.makedirs(MODELS, exist_ok=True)

keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)   # CATAT versi ini untuk requirements.txt

## 1. Muat data

In [ ]:
sim = pd.read_excel(DATA_XLSX, sheet_name='Data_Training')
asli = pd.read_excel(DATA_XLSX, sheet_name='Sheet').assign(Cycle_ID=0, Split='test')

CLASS_NAMES = ['masa_pertumbuhan', 'mendekati_panen', 'siap_panen']
print('Data_Training:', sim.shape, '| Data asli:', asli.shape)
display(sim.head())
print(sim['Split'].value_counts() / 120, '(jumlah siklus per split)')
print(sim['Stage_ID'].value_counts().sort_index())

## 2. Fitur dan normalisasi
Fitur: kelembapan, suhu (dinormalisasi 0-1 memakai min/max **data latih saja**), ditambah one-hot jam (08:00 / 12:00 / 16:00).
Kolom `Day`, `DAP`, `Time_Index` **tidak dipakai** sebagai fitur supaya model belajar dari sensor, bukan dari nomor hari.

In [ ]:
SENSOR = ['Soil Moisture (%)', 'Temperature (°C)']
JAM = {'08:00': 0, '12:00': 1, '16:00': 2}

train_df = sim[sim.Split == 'train']
mins = train_df[SENSOR].min().values.astype('float32')
maxs = train_df[SENSOR].max().values.astype('float32')
print('min:', mins, 'max:', maxs)

def make_features(df):
    s = (df[SENSOR].values.astype('float32') - mins) / (maxs - mins)
    jam = np.eye(3, dtype='float32')[df['Time'].astype(str).map(JAM).values.astype(int)]
    return np.hstack([s, jam])          # (n, 5)

## 3. Membuat jendela (sliding window)
- Klasifikasi: 6 pembacaan terakhir -> tahap pada pembacaan terakhir.
- Prediksi: 12 pembacaan terakhir -> 12 pembacaan berikutnya (kelembapan, suhu, maturity).

Jendela dibuat **di dalam satu siklus saja** dan setiap siklus hanya masuk ke satu split, jadi tidak ada kebocoran data.

In [ ]:
W_CLS, W_IN, H_OUT = 6, 12, 12

def build_windows(df):
    Xc, yc, spc, Xf, yf, spf = [], [], [], [], [], []
    for cid, g in df.groupby('Cycle_ID'):
        g = g.sort_values('Time_Index')
        X = make_features(g)
        stage = g['Stage_ID'].values
        T = np.hstack([
            (g[SENSOR].values.astype('float32') - mins) / (maxs - mins),
            g['Ground Truth Maturity Level (%)'].values.astype('float32')[:, None] / 100.0,
        ])
        split = g['Split'].iloc[0]
        for i in range(W_CLS - 1, len(g)):
            Xc.append(X[i - W_CLS + 1:i + 1]); yc.append(stage[i]); spc.append(split)
        for i in range(W_IN, len(g) - H_OUT + 1):
            Xf.append(X[i - W_IN:i]); yf.append(T[i:i + H_OUT]); spf.append(split)
    return (np.array(Xc), np.array(yc), np.array(spc),
            np.array(Xf), np.array(yf), np.array(spf))

Xc, yc, spc, Xf, yf, spf = build_windows(sim)
Xc_o, yc_o, _, Xf_o, yf_o, _ = build_windows(asli)

def part(X, y, sp, name):
    m = sp == name
    return X[m], y[m]

Xc_tr, yc_tr = part(Xc, yc, spc, 'train'); Xc_va, yc_va = part(Xc, yc, spc, 'val'); Xc_te, yc_te = part(Xc, yc, spc, 'test')
Xf_tr, yf_tr = part(Xf, yf, spf, 'train'); Xf_va, yf_va = part(Xf, yf, spf, 'val'); Xf_te, yf_te = part(Xf, yf, spf, 'test')
print('Klasifikasi  train/val/test/asli:', len(Xc_tr), len(Xc_va), len(Xc_te), len(Xc_o))
print('Prediksi     train/val/test/asli:', len(Xf_tr), len(Xf_va), len(Xf_te), len(Xf_o))

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=yc_tr)
class_weight = dict(enumerate(cw))
print('class_weight:', class_weight)

## 4. Model klasifikasi (LSTM)

In [ ]:
def build_classifier():
    inp = keras.Input(shape=(W_CLS, 5))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(3, activation='softmax')(x)
    m = keras.Model(inp, out, name='lstm_klasifikasi')
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

clf = build_classifier()
clf.summary()

cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
      keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)]
hist = clf.fit(Xc_tr, yc_tr, validation_data=(Xc_va, yc_va), epochs=100, batch_size=64,
               class_weight=class_weight, callbacks=cb, verbose=2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history['loss'], label='train'); ax[0].plot(hist.history['val_loss'], label='val'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(hist.history['accuracy'], label='train'); ax[1].plot(hist.history['val_accuracy'], label='val'); ax[1].set_title('Akurasi'); ax[1].legend()
plt.show()

## 5. Evaluasi klasifikasi
Dua ujian: (a) data uji simulasi (siklus yang belum pernah dilihat model) dan (b) **data asli** sebagai uji tambahan.
Skor di data asli biasanya lebih rendah karena polanya sedikit berbeda. Itu wajar, laporkan keduanya dengan jujur.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate(X, y, title):
    p = clf.predict(X, verbose=0).argmax(axis=1)
    acc = float((p == y).mean())
    print(f'=== {title}: akurasi = {acc:.3f} ===')
    print(classification_report(y, p, labels=[0, 1, 2], target_names=CLASS_NAMES, digits=3, zero_division=0))
    cm = confusion_matrix(y, p, labels=[0, 1, 2])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap='Greens', xticks_rotation=20)
    plt.title(title); plt.show()
    rep = classification_report(y, p, labels=[0, 1, 2], target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    return {'akurasi': acc, 'cm': cm.tolist(), 'report': rep}

ev_test = evaluate(Xc_te, yc_te, 'Uji simulasi (siklus belum dilihat)')
ev_asli = evaluate(Xc_o, yc_o, 'Uji data ASLI')
acc_test, acc_asli = ev_test['akurasi'], ev_asli['akurasi']

## 6. Model prediksi 4 hari ke depan
Masukan: 12 pembacaan terakhir. Keluaran: 12 pembacaan berikutnya x 3 nilai (kelembapan, suhu, maturity).
Tahap tumbuh untuk hari-hari ke depan diturunkan dari **maturity prediksi** memakai ambang di bawah.

In [ ]:
def build_forecaster():
    inp = keras.Input(shape=(W_IN, 5))
    x = layers.LSTM(96, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(48)(x)
    x = layers.Dense(96, activation='relu')(x)
    x = layers.Dense(H_OUT * 3)(x)
    out = layers.Reshape((H_OUT, 3))(x)
    m = keras.Model(inp, out, name='lstm_prediksi')
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

fc = build_forecaster()
cb2 = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
       keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6)]
hist2 = fc.fit(Xf_tr, yf_tr, validation_data=(Xf_va, yf_va), epochs=150, batch_size=64, callbacks=cb2, verbose=2)

In [ ]:
def mae_units(X, y, title):
    p = fc.predict(X, verbose=0)
    err = np.abs(p - y).mean(axis=(0, 1))          # rata-rata per target (skala 0-1)
    mae_moist = err[0] * (maxs[0] - mins[0])
    mae_temp = err[1] * (maxs[1] - mins[1])
    mae_mat = err[2] * 100
    print(f'{title}: MAE kelembapan = {mae_moist:.2f} %, suhu = {mae_temp:.2f} C, maturity = {mae_mat:.2f} %')
    return dict(moisture=float(mae_moist), temperature=float(mae_temp), maturity=float(mae_mat))

mae_test = mae_units(Xf_te, yf_te, 'Uji simulasi')
mae_orig = mae_units(Xf_o, yf_o, 'Data ASLI   ')

# ambang maturity -> tahap (dari rentang maturity tiap tahap di Data_Training)
print(sim.groupby('Stage_ID')['Ground Truth Maturity Level (%)'].agg(['min', 'max']))
THRESH = [66.5, 89.0]    # <66.5 masa_pertumbuhan | <89 mendekati_panen | selain itu siap_panen

In [ ]:
# contoh visual: prediksi vs sebenarnya untuk satu jendela uji
i = len(Xf_te) // 2
p = fc.predict(Xf_te[i:i + 1], verbose=0)[0]
t = yf_te[i]
names = ['Kelembapan', 'Suhu', 'Maturity']
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for k in range(3):
    ax[k].plot(t[:, k], 'o-', label='sebenarnya'); ax[k].plot(p[:, k], 's--', label='prediksi')
    ax[k].set_title(names[k] + ' (skala 0-1)'); ax[k].legend()
plt.show()

## 7. Simpan model
Tiga file dihasilkan di `MyDrive/pakcoy-project/models/`:
`model_lstm_klasifikasi.keras`, `model_lstm_forecast.keras`, `config_lstm.json`.
**Unduh ketiganya** dan taruh di folder `models/` pada project Streamlit.

In [ ]:
clf.save(f'{MODELS}/model_lstm_klasifikasi.keras')
fc.save(f'{MODELS}/model_lstm_forecast.keras')

config = {
    'tf_version': tf.__version__,
    'class_names': CLASS_NAMES,
    'sensor_cols': SENSOR,
    'mins': [float(v) for v in mins],
    'maxs': [float(v) for v in maxs],
    'jam_map': JAM,
    'w_cls': W_CLS, 'w_in': W_IN, 'h_out': H_OUT,
    'maturity_thresholds': THRESH,
    'metrics': {
        'akurasi_uji_simulasi': acc_test,
        'akurasi_data_asli': acc_asli,
        'mae_forecast_uji_simulasi': mae_test,
        'mae_forecast_data_asli': mae_orig,
    },
    'evaluasi': {'Uji simulasi': ev_test, 'Data asli': ev_asli},
}
with open(f'{MODELS}/config_lstm.json', 'w') as f:
    json.dump(config, f, indent=2, default=float)

print(os.listdir(MODELS))
print('Versi TensorFlow untuk requirements.txt:', tf.__version__)